# RankingRecommender + XGBoost on MovieLens 1M

End-to-end demo of `RankingRecommender` with `XGBClassifierEstimator` on the MovieLens 1M dataset.

**Evaluation protocol**: leave-last-out — for each user, the last *positive* interaction
(rating ≥ 4, sorted by timestamp) is the test item; all other interactions (positive and negative)
form the training set.

**Why all ratings (not just positives)?**  
XGBoost is a feature-based pointwise ranker. Low ratings are genuine negative signal — they tell the
model what a user *dislikes*, which is just as informative as what they like. Including them
sharpens the decision boundary. Sequential models like SASRec sample random negatives instead
because they model order, not explicit preferences.

**Metrics**: HR@10 (Hit Rate) and NDCG@10.

**Data**: Downloaded automatically to `nb_examples/data/ml-1m/` (excluded from git).

## 1. Imports

In [ ]:
import logging
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from skrec.dataset.interactions_dataset import InteractionsDataset
from skrec.dataset.items_dataset import ItemsDataset
from skrec.estimator.classification.xgb_classifier import XGBClassifierEstimator
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.scorer.universal import UniversalScorer

# Suppress verbose INFO logs from the library's internal loggers
logging.basicConfig(level=logging.WARNING)
logging.getLogger("recommender").setLevel(logging.WARNING)

RAW_DIR = Path("data/movielens-1m")
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data/ranking-xgboost")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Imports OK")

## 2. Download MovieLens 1M

In [ ]:
ML1M_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = RAW_DIR / "ml-1m.zip"

if not (RAW_DIR / "ratings.dat").exists():
    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(ML1M_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            if name.endswith(".dat"):
                filename = Path(name).name
                with zf.open(name) as src, open(RAW_DIR / filename, "wb") as dst:
                    dst.write(src.read())
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

## 3. Load and Preprocess

MovieLens 1M has three data files:
- **ratings.dat** — `UserID::MovieID::Rating::Timestamp`
- **movies.dat** — `MovieID::Title::Genres` (pipe-separated genre tags)
- **users.dat** — `UserID::Gender::Age::Occupation::Zip-code`

We use all three. User demographics and movie genres give XGBoost a richer signal
than a pure ID-based collaborative filter.

In [ ]:
# ratings.dat: UserID::MovieID::Rating::Timestamp
ratings = pd.read_csv(
    RAW_DIR / "ratings.dat",
    sep="::",
    engine="python",
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

# movies.dat: MovieID::Title::Genres
movies = pd.read_csv(
    RAW_DIR / "movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1",
)

# users.dat: UserID::Gender::Age::Occupation::Zip-code
users_raw = pd.read_csv(
    RAW_DIR / "users.dat",
    sep="::",
    engine="python",
    names=["UserID", "Gender", "Age", "Occupation", "Zip"],
)

print(f"Ratings : {len(ratings):,}  |  Users: {ratings.UserID.nunique():,}  |  Movies: {ratings.MovieID.nunique():,}")
print(f"Movies  : {len(movies):,}  |  Genres: {movies.Genres.str.split('|').explode().nunique()}")
print(f"Users   : {len(users_raw):,}")
ratings.head(3)

## 4. Feature Engineering

### Item features — genre one-hot encoding

Each movie can have multiple genres (e.g. `"Action|Adventure|Sci-Fi"`). We one-hot encode
them so XGBoost can learn which genres a user prefers.

### User features — demographics

We encode the three demographic columns into the **interactions** DataFrame so that
at inference time, the scorer can replicate user context across all candidate items
without needing a separate user lookup at scoring time.

| Feature | Encoding |
|---|---|
| Gender | Binary: `gender_M` (1=Male, 0=Female) |
| Age | Ordinal numeric (7 buckets: 1, 18, 25, 35, 45, 50, 56) |
| Occupation | Numeric code 0–20 (as-is from ML-1M) |

In [ ]:
# --- Item features: genre one-hot ---
all_genres = sorted(
    {g for genres in movies.Genres.str.split("|") for g in genres}
)
print(f"{len(all_genres)} unique genres: {all_genres}")

genre_dummies = movies.Genres.str.get_dummies(sep="|").add_prefix("genre_")
items_df = pd.concat(
    [movies[["MovieID"]].rename(columns={"MovieID": "ITEM_ID"}), genre_dummies], axis=1
)
items_df["ITEM_ID"] = items_df["ITEM_ID"].astype(str)

print(f"\nItems DataFrame: {items_df.shape[0]:,} rows × {items_df.shape[1]} columns")
items_df.head(3)

In [ ]:
# --- User-demographic features ---
users_feat = users_raw[["UserID", "Gender", "Age", "Occupation"]].copy()
users_feat["gender_M"] = (users_feat["Gender"] == "M").astype(int)
users_feat = users_feat.rename(columns={"Age": "age", "Occupation": "occupation"})
users_feat = users_feat[["UserID", "gender_M", "age", "occupation"]]

# --- Build interactions: all ratings, binary label ---
# Rating ≥ 4 → positive (1); rating < 4 → negative (0).
# Using observed negatives (low-rated movies) gives XGBoost real
# dislike signal. This contrasts with SASRec which samples random negatives.
interactions = ratings.merge(users_feat, on="UserID", how="left")

interactions = pd.DataFrame({
    "USER_ID":     interactions["UserID"].astype(str),
    "ITEM_ID":     interactions["MovieID"].astype(str),
    "OUTCOME":     (interactions["Rating"] >= 4).astype(float),
    "TIMESTAMP":   interactions["Timestamp"],
    "gender_M":    interactions["gender_M"],
    "age":         interactions["age"],
    "occupation":  interactions["occupation"],
})

print(f"Interactions: {len(interactions):,}  |  "
      f"positive rate: {interactions.OUTCOME.mean():.1%}")
interactions.head(3)

## 5. Train / Test Split (Leave-Last-Positive-Out)

For each user, the **last positive interaction** (rating ≥ 4, by timestamp) is the test item.
Everything else — both liked and disliked interactions — is used for training.
Users with fewer than 3 total interactions are excluded.

**Evaluation**: sampled ranking — the test item is ranked against 100 randomly sampled
items the user has never interacted with (101 candidates total).

In [ ]:
interactions = interactions.sort_values(["USER_ID", "TIMESTAMP"]).reset_index(drop=True)

# Keep users with at least 3 interactions
user_counts = interactions.groupby("USER_ID").size()
valid_users = user_counts[user_counts >= 3].index
interactions = interactions[interactions["USER_ID"].isin(valid_users)].reset_index(drop=True)

# Find the last POSITIVE interaction per user → test item
pos_interactions = interactions[interactions["OUTCOME"] == 1.0].copy()
last_pos_idx = pos_interactions.groupby("USER_ID")["TIMESTAMP"].idxmax()
test_df = pos_interactions.loc[last_pos_idx].reset_index(drop=True)

# Training: everything except each user's test row
train_df = interactions.drop(index=last_pos_idx).reset_index(drop=True)

# Only keep test users that also appear in training (they need features)
test_df = test_df[test_df["USER_ID"].isin(train_df["USER_ID"].unique())].reset_index(drop=True)

print(f"Train : {len(train_df):,} interactions  (positive rate: {train_df.OUTCOME.mean():.1%})")
print(f"Test  : {len(test_df):,} interactions  (one per user)")
print(f"Users : {train_df.USER_ID.nunique():,} in train  |  {test_df.USER_ID.nunique():,} in test")

## 6. Why `UniversalScorer`?

The library offers several scorers. For XGBoost-based ranking, `UniversalScorer` is the
right choice. Here's the decision logic:

| Scorer | How it works | Why **not** here |
|---|---|---|
| **`UniversalScorer`** ✓ | Concatenates user + item features → one XGBoost call across all items | **This one.** |
| `IndependentScorer` | Trains a **separate** model per item | 3,706 XGBoost models for ML-1M — impractical. Only works for small item sets (e.g. 5–50 items). |
| `MulticlassScorer` | One multi-class model where each class = an item | Requires all items as classes at training time; doesn't generalise to new items and is memory-intensive at this catalog scale. |
| `SequentialScorer` | Scores via a sequence model (SASRec / HRNN) | For sequential/session-based models only; not compatible with XGBoost. |

**What `UniversalScorer` does at training time:**
```
interactions (USER_ID, ITEM_ID, OUTCOME, user_features)
    ↓ join on ITEM_ID
items (ITEM_ID, genre_features)
    ↓
X = [user_features | genre_features]   y = OUTCOME
    ↓
XGBoost.fit(X, y)
```

**What `UniversalScorer` does at inference time (scoring all items for a user):**
```
query (USER_ID, user_features)   ← one row per user
    ↓ replicate for each item
[user_features | genre_item_1],
[user_features | genre_item_2],
...                              ← n_items rows
    ↓
XGBoost.predict_proba(...)       ← one batched call
    ↓
scores[user, :] shape (n_items,)
```

**Key advantage over collaborative filtering**: XGBoost can leverage *side features*.
A user's age and occupation interact with item genres — a 25-year-old may respond
differently to sci-fi than a 50-year-old. Pure CF (matrix factorization) ignores this.

## 7. Save CSVs and Create Datasets

In [ ]:
train_path = str(DATA_DIR / "train_interactions.csv")
items_path = str(DATA_DIR / "items.csv")

# Drop TIMESTAMP before saving — not a model feature, just used for ordering
if not Path(train_path).exists():
    train_df.drop(columns=["TIMESTAMP"]).to_csv(train_path, index=False)
if not Path(items_path).exists():
    items_df.to_csv(items_path, index=False)

interactions_ds = InteractionsDataset(data_location=train_path)
items_ds        = ItemsDataset(data_location=items_path)

print(f"Training data : {len(train_df):,} interactions")
print(f"Item catalog  : {len(items_df):,} movies × {items_df.shape[1]-1} genre features")
print("Datasets created.")

## 8. Build and Train

The training matrix has shape `(n_train_interactions, n_user_features + n_genre_features)`.
With ~975K rows and 21 features, XGBoost trains in seconds.

In [ ]:
estimator = XGBClassifierEstimator(
    params={
        "n_estimators":     300,
        "max_depth":        6,
        "learning_rate":    0.1,
        "subsample":        0.8,
        "colsample_bytree": 0.8,
        "eval_metric":      "logloss",
        "random_state":     42,
        "n_jobs":           -1,
    }
)

scorer      = UniversalScorer(estimator)
recommender = RankingRecommender(scorer)

print("Training XGBoost...")
recommender.train(
    interactions_ds=interactions_ds,
    items_ds=items_ds,
)
print(f"Training complete. Features: {estimator.feature_names}")

## 9. Feature Importance

One of XGBoost's advantages over neural CF is interpretability.
Feature importances reveal which user attributes and item genres drive ranking.

In [ ]:
import matplotlib.pyplot as plt

importances = estimator._model.get_booster().get_score(importance_type="gain")
imp_df = (
    pd.DataFrame.from_dict(importances, orient="index", columns=["gain"])
    .sort_values("gain", ascending=True)
)

fig, ax = plt.subplots(figsize=(8, max(4, len(imp_df) * 0.35)))
ax.barh(imp_df.index, imp_df["gain"])
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost feature importances")
plt.tight_layout()
plt.show()

print("\nTop-5 features by gain:")
print(imp_df.tail(5).to_string())

## 10. Evaluate: HR@10 and NDCG@10 (Sampled Ranking)

For each user, the held-out test item is ranked against **100 randomly sampled negative
items** (items the user has never interacted with). HR@10 and NDCG@10 are computed within
these 101 candidates.

We score all test users in a **single batched call** to `scorer.score_items()` which builds
one `(n_users × n_items)` feature matrix and runs XGBoost predict once.

### What to expect

The model here only uses **static demographics** (gender, age, occupation) and **item genres**.
It has **no interaction history features** — it doesn't know which specific movies a user has
liked before. This means it can only learn coarse demographic → genre preferences, not individual
user tastes.

| Model | HR@10 | What it uses |
|---|---|---|
| Random baseline | ~0.099 (10/101) | Nothing |
| **This notebook (XGBoost + demographics + genres)** | **~0.18–0.22** | Gender, age, occupation, genres |
| SASRec (positives-only, same eval protocol) | ~0.89 | Full interaction history |

XGBoost's advantage over collaborative filtering is **side-feature utilisation** — when you have
rich per-user or per-item features beyond IDs. A stronger version of this model would add
history-derived features (e.g. each user's per-genre engagement rate, recency, item popularity)
to close the gap with sequence models. The goal of this notebook is to show the API wiring, not
to maximise the metric.

> **Comparability note**: the SASRec number is not directly comparable — SASRec sees the full
> rated sequence; this XGBoost model sees only three demographic fields per user.

In [ ]:
rng = np.random.default_rng(42)
all_item_ids = scorer.item_names  # sorted array of all known item IDs

# Only evaluate users whose test item is in the item vocabulary
known_items = set(scorer.item_names)
eval_test_df = test_df[test_df["ITEM_ID"].isin(known_items)].copy()
eval_user_ids = eval_test_df["USER_ID"].unique().tolist()

print(f"Evaluating {len(eval_user_ids):,} users ...")

# Build one query row per test user (with user demographics, no ITEM_ID)
# The scorer will replicate these features across all items via _replicate_for_items.
# We take each user's LAST training interaction as the query context so that
# user demographic features (gender, age, occupation) are present in the row.
last_train = (
    train_df.sort_values("TIMESTAMP")
    .groupby("USER_ID")
    .last()
    .reset_index()[["USER_ID", "gender_M", "age", "occupation"]]
)
query_df = last_train[last_train["USER_ID"].isin(eval_user_ids)].reset_index(drop=True)

# Score ALL items for all test users in one batched call — shape (n_users, n_items)
all_scores_df = scorer.score_items(interactions=query_df)
user_order = query_df["USER_ID"].tolist()
item_name_to_idx = {name: i for i, name in enumerate(scorer.item_names)}

# Ground-truth and seen-items lookup for negative sampling
gt_lookup = eval_test_df.set_index("USER_ID")["ITEM_ID"].to_dict()
user_seen = (
    train_df.groupby("USER_ID")["ITEM_ID"]
    .apply(set)
    .to_dict()
)

TOP_K       = 10
N_NEGATIVES = 100

hits, ndcgs = [], []
scores_np = all_scores_df.to_numpy()  # (n_users, n_items)

for i, user_id in enumerate(user_order):
    test_item = gt_lookup.get(user_id)
    if test_item is None or test_item not in item_name_to_idx:
        continue

    seen = user_seen.get(user_id, set())
    unseen = all_item_ids[~np.isin(all_item_ids, list(seen))]
    neg_sample = rng.choice(unseen, size=min(N_NEGATIVES, len(unseen)), replace=False)

    candidate_ids = [test_item] + list(neg_sample)
    candidate_idxs = [item_name_to_idx[c] for c in candidate_ids if c in item_name_to_idx]
    candidate_scores = scores_np[i, candidate_idxs]

    test_score = scores_np[i, item_name_to_idx[test_item]]
    rank = int((candidate_scores > test_score).sum()) + 1  # 1-indexed

    hits.append(1 if rank <= TOP_K else 0)
    ndcgs.append(1.0 / np.log2(rank + 1) if rank <= TOP_K else 0.0)

print(f"\n{'='*40}")
print(f"Evaluation: 1 positive + {N_NEGATIVES} random negatives")
print(f"HR@{TOP_K}   : {np.mean(hits):.4f}")
print(f"NDCG@{TOP_K} : {np.mean(ndcgs):.4f}")
print(f"Users evaluated: {len(hits):,}")
print(f"{'='*40}")

## 11. Sample Recommendations

Show top-10 full-catalog recommendations for a few users alongside their held-out test item.
We pass `interactions=query_df` (user demographics) so the scorer can build the feature matrix.

In [ ]:
movie_title = movies.set_index(movies["MovieID"].astype(str))["Title"].to_dict()

sample_users = user_order[:5]
sample_query = query_df[query_df["USER_ID"].isin(sample_users)].reset_index(drop=True)
sample_user_order = sample_query["USER_ID"].tolist()

top_k_recs = recommender.recommend(interactions=sample_query, top_k=TOP_K)

for i, user_id in enumerate(sample_user_order):
    recs = list(top_k_recs[i])
    test_item = gt_lookup.get(user_id, "?")
    hit = "HIT" if test_item in recs else "MISS"
    age_val = int(query_df[query_df.USER_ID == user_id]["age"].iloc[0])
    gender_val = "M" if int(query_df[query_df.USER_ID == user_id]["gender_M"].iloc[0]) == 1 else "F"
    print(f"\nUser {user_id} ({gender_val}, age {age_val})  |  Test: {movie_title.get(test_item, test_item)}  [{hit}]")
    for rank, item_id in enumerate(recs, 1):
        marker = " <-- TEST ITEM" if item_id == test_item else ""
        print(f"  {rank:2}. {movie_title.get(item_id, item_id)}{marker}")